# 🧪 Human-in-the-Loop (HIL) với DeepSeek Agent

Notebook này demo cách tích hợp **Human-in-the-Loop** vào agent workflow sử dụng **DeepSeek API** + **LangGraph**.

## Mục lục

1. **Import thư viện** — langgraph, langchain, IPython widgets
2. **Cài đặt dependencies** — pip install
3. **Định nghĩa Tools** — Tool gửi email, tool tra cứu
4. **Xây dựng StateGraph với HIL** — interrupt_before, interrupt_after
5. **Approval Pattern** — Human duyệt trước khi tool chạy
6. **Log & Phân tích** — Audit trail, thống kê

In [1]:
# Cell này cài đặt các thư viện cần thiết
# Chạy cell này trước khi chạy các cell khác

import subprocess
import sys

deps = [
    "langgraph>=0.4.0",
    "langchain-core>=0.3.0",
    "langchain-community>=0.3.0",
    "langchain-openai>=0.3.0",
    "ipywidgets>=8.0.0",
    "openai>=1.0.0",
]

for dep in deps:
    print(f"Installing {dep}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])
print("✅ All dependencies installed!")


Installing langgraph>=0.4.0...
Installing langchain-core>=0.3.0...
Installing langchain-community>=0.3.0...
Installing langchain-openai>=0.3.0...
Installing ipywidgets>=8.0.0...
Installing openai>=1.0.0...
✅ All dependencies installed!


## 1. Import Thư Viện

Import các thư viện chính: LangGraph cho StateGraph, LangChain Core cho tools, IPython widgets cho giao diện HIL.

In [ ]:
import os
import json
from datetime import datetime
from typing import TypedDict, Literal, Optional, Annotated
from typing_extensions import TypedDict

# LangGraph
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

# LangChain
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_openai import ChatOpenAI

# IPython Widgets cho HIL
import ipywidgets as widgets
from IPython.display import display, clear_output

print("✅ Import thành công!")


## 2. Cấu Hình DeepSeek API Client

Cấu hình ChatOpenAI để gọi **DeepSeek API** (tương thích OpenAI format).
Bạn cần có `DEEPSEEK_API_KEY` trong environment variable hoặc nhập trực tiếp.

In [ ]:
# Cấu hình DeepSeek API
# Lấy API key từ environment variable hoặc nhập tay
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")

if not DEEPSEEK_API_KEY:
    print("⚠️  Chưa có DEEPSEEK_API_KEY. Vui lòng nhập ở cell dưới.")
else:
    print(f"✅ DeepSeek API Key found: ...{DEEPSEEK_API_KEY[-4:]}")

# Khởi tạo LLM với DeepSeek (OpenAI-compatible)
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com/v1",
    temperature=0.7,
)

print(f"✅ LLM configured: deepseek-chat")

In [ ]:
# Nhập API Key nếu chưa có
if not DEEPSEEK_API_KEY:
    api_key_input = widgets.Password(
        description="API Key:",
        placeholder="sk-...",
        layout=widgets.Layout(width="500px")
    )
    submit_btn = widgets.Button(description="✅ Set Key", button_style="primary")
    output = widgets.Output()

    def set_key(b):
        global DEEPSEEK_API_KEY, llm
        DEEPSEEK_API_KEY = api_key_input.value
        os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY
        llm = ChatOpenAI(
            model="deepseek-chat",
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com/v1",
            temperature=0.7,
        )
        with output:
            print(f"✅ API Key set: ...{DEEPSEEK_API_KEY[-4:]}")

    submit_btn.on_click(set_key)
    display(widgets.VBox([api_key_input, submit_btn, output]))
else:
    print("✅ API Key already configured.")

## 3. Định Nghĩa Tools & State

Định nghĩa các **tools** mà agent có thể gọi, và **State** cho graph.
Tools sẽ là nơi chúng ta cài đặt **interrupt points** cho HIL.

In [ ]:
from langchain_core.tools import tool


# --- Tools mà Agent có thể gọi ---

@tool
def search_database(query: str) -> str:
    """Search customer database for information."""
    print(f"🔍 [TOOL] search_database: {query}")
    # Mock data
    data = {
        "john@example.com": {"name": "John Doe", "plan": "Premium", "status": "active"},
        "jane@example.com": {"name": "Jane Smith", "plan": "Basic", "status": "inactive"},
    }
    for email, info in data.items():
        if email in query.lower() or info["name"].lower() in query.lower():
            return json.dumps(info, indent=2)
    return json.dumps({"error": "Customer not found"})


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to recipient. REQUIRES HUMAN APPROVAL."""
    print(f"📧 [TOOL] send_email to={to}, subject={subject}")
    # Trong thực tế sẽ gọi SMTP, ở đây mock
    return f"✅ Email sent to {to}: {subject}"


@tool
def generate_report(report_type: str, params: str) -> str:
    """Generate a report. REQUIRES HUMAN APPROVAL if report_type is 'delete' or 'financial'."""
    print(f"📊 [TOOL] generate_report: {report_type}")
    return f"📄 Report '{report_type}' generated with params: {params}"


tools = [search_database, send_email, generate_report]
print(f"✅ {len(tools)} tools defined: {[t.name for t in tools]}")


## 4. Xây Dựng StateGraph với HIL

Đây là phần quan trọng nhất: chúng ta build **LangGraph StateGraph** với:

- `interrupt_before=["tools"]` — Dừng **trước khi** agent gọi tool
- `interrupt_after=["agent"]` — Dừng **sau khi** agent trả lời
- **Checkpointer** — Lưu trạng thái để resume sau

In [ ]:
# Bind tools vào LLM
llm_with_tools = llm.bind_tools(tools)


# Node: Agent
def agent_node(state: MessagesState) -> dict:
    """Agent node: nhận messages và gọi LLM."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# Node: Tool Executor
tool_node = ToolNode(tools)


# Build Graph
graph = StateGraph(MessagesState)

graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges(
    "agent",
    tools_condition,  # Nếu agent gọi tool → tools, nếu không → END
)
graph.add_edge("tools", "agent")

# Compile với HIL!
checkpointer = MemorySaver()

app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["tools"],  # ⛔ Dừng TRƯỚC khi gọi tool → chờ human approve
    # interrupt_after=["agent"], # (Có thể thêm) Dừng sau khi agent suy nghĩ
)

print("✅ Graph compiled with HIL!")
print("Interrupt points: before 'tools' node")


## 5. Chạy Agent với HIL — Ví Dụ Cơ Bản

Chạy agent với một câu hỏi. Agent sẽ dừng lại trước khi gọi tool,
chờ bạn xác nhận (approve/reject) trước khi tiếp tục.

In [ ]:
# --- Thread ID để duy trì state ---
thread_id = "hil-demo-1"
config = {"configurable": {"thread_id": thread_id}}

# --- Gửi câu hỏi ---
user_input = "Search for customer John Doe in database, then send him an email with subject 'Welcome' and body 'Hello John!'"

print(f"👤 User: {user_input}")
print("─" * 60)

# Stream cho đến khi interrupt hoặc kết thúc
for event in app.stream(
    {"messages": [HumanMessage(content=user_input)]},
    config=config,
    stream_mode="values",
):
    msg = event["messages"][-1]
    if isinstance(msg, AIMessage) and msg.content:
        print(f"🤖 Agent: {msg.content[:200]}...")
    elif isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🔧 Agent muốn gọi: {tc['name']}({tc['args']})")

# --- Kiểm tra state sau interrupt ---
state = app.get_state(config)
print("─" * 60)
print(f"⛔ Agent đã dừng tại interrupt!")
print(f"📌 Next node: {state.next}")
print(f"📌 State: {'⏸️ PAUSED' if state.next else '✅ COMPLETED'}")

# Hiển thị các tool call đang chờ duyệt
last_msg = state.values["messages"][-1]
if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
    print("\n📋 Các tool call chờ duyệt:")
    for i, tc in enumerate(last_msg.tool_calls, 1):
        print(f"  {i}. {tc['name']}({json.dumps(tc['args'], indent=4)})")


## 6. Human Review — Dùng Widgets để Approve/Reject

Sử dụng **IPython widgets** để tạo giao diện duyệt tool call.
Bạn có thể **Approve** (tiếp tục) hoặc **Reject** (hủy) từng tool call.

In [ ]:
def create_hil_panel(config, app):
    """Tạo panel HIL với Approve/Reject buttons."""
    state = app.get_state(config)
    last_msg = state.values["messages"][-1]

    if not state.next:
        print("✅ Agent đã hoàn thành, không có tool nào chờ duyệt.")
        return None

    if not (hasattr(last_msg, "tool_calls") and last_msg.tool_calls):
        print("ℹ️  Không có tool call nào để duyệt.")
        return None

    output_area = widgets.Output()
    log_area = widgets.Output()

    with output_area:
        clear_output(wait=True)
        print("📋 **Tool Calls Chờ Duyệt:**")
        for i, tc in enumerate(last_msg.tool_calls, 1):
            print(f"\n  **{i}. {tc['name']}**")
            print(f"     Args: {json.dumps(tc['args'], indent=6)}")
        print("\n" + "─" * 50)

    # Buttons
    btn_approve = widgets.Button(
        description="✅ Approve All",
        button_style="success",
        layout=widgets.Layout(width="150px")
    )
    btn_reject = widgets.Button(
        description="❌ Reject All",
        button_style="danger",
        layout=widgets.Layout(width="150px")
    )
    btn_status = widgets.Button(
        description="🔍 Check Status",
        button_style="info",
        layout=widgets.Layout(width="150px")
    )

    def on_approve(b):
        with log_area:
            clear_output(wait=True)
            print("✅ **Approved!** Resuming agent...\n")
            try:
                # Resume: gọi None để tiếp tục từ interrupt
                for event in app.stream(None, config=config, stream_mode="values"):
                    msg = event["messages"][-1]
                    if isinstance(msg, AIMessage) and msg.content:
                        print(f"🤖 Agent: {msg.content}")
                    elif isinstance(msg, ToolMessage):
                        print(f"🔧 Tool result: {msg.content[:100]}...")
                print(f"\n✅ Agent hoàn thành!")
            except Exception as e:
                print(f"❌ Error: {e}")

    def on_reject(b):
        with log_area:
            clear_output(wait=True)
            print("❌ **Rejected!** Gửi feedback cho agent...\n")
            try:
                # Gửi message từ human thay vì resume tool
                app.update_state(
                    config,
                    {"messages": [HumanMessage(
                        content="I reject those tool calls. Please explain why you wanted to call them and suggest an alternative."
                    )]},
                )
                # Resume để agent xử lý feedback
                for event in app.stream(None, config=config, stream_mode="values"):
                    msg = event["messages"][-1]
                    if isinstance(msg, AIMessage) and msg.content:
                        print(f"🤖 Agent: {msg.content}")
                print(f"\n✅ Agent đã nhận feedback!")
            except Exception as e:
                print(f"❌ Error: {e}")

    def on_status(b):
        with log_area:
            clear_output(wait=True)
            s = app.get_state(config)
            print(f"📌 Thread: {config['configurable']['thread_id']}")
            print(f"📌 Next node: {s.next}")
            print(f"📌 Messages: {len(s.values.get('messages', []))}")
            print(f"📌 Is interrupted: {s.next is not None and len(s.next) > 0}")

    btn_approve.on_click(on_approve)
    btn_reject.on_click(on_reject)
    btn_status.on_click(on_status)

    controls = widgets.HBox([btn_approve, btn_reject, btn_status])
    panel = widgets.VBox([
        widgets.HTML("<h3>🧑‍💼 Human Review Panel</h3>"),
        output_area,
        controls,
        log_area,
    ])
    return panel


# Hiển thị HIL Panel
panel = create_hil_panel(config, app)
if panel:
    display(panel)
else:
    print("Agent đã hoàn thành. Chạy cell 5 để thử lại.")


## 7. Advanced: Conditional Interrupt (Chỉ Dừng Tool Nguy Hiểm)

Thay vì dừng *tất cả* tool, ta có thể dùng **custom logic** để chỉ dừng
các tool "nguy hiểm" (send_email, delete_data) — còn tool "an toàn" (search) chạy tự do.

In [ ]:
# --- Tools cần approval ---
DANGEROUS_TOOLS = {"send_email", "generate_report"}


def should_interrupt(state: MessagesState) -> bool:
    """Kiểm tra xem có cần interrupt không."""
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            if tc["name"] in DANGEROUS_TOOLS:
                return True  # Cần human approve
    return False  # Cho chạy tự do


# Build graph mới với conditional interrupt
graph_v2 = StateGraph(MessagesState)
graph_v2.add_node("agent", agent_node)
graph_v2.add_node("tools", tool_node)
graph_v2.add_edge(START, "agent")
graph_v2.add_conditional_edges("agent", tools_condition)
graph_v2.add_edge("tools", "agent")

checkpointer_v2 = MemorySaver()

# Dùng interrupt_before=["tools"] + tự kiểm tra trong node
app_v2 = graph_v2.compile(
    checkpointer=checkpointer_v2,
    interrupt_before=["tools"],
)

print("✅ Graph v2 compiled!")
print(f"Dangerous tools (cần approval): {DANGEROUS_TOOLS}")


In [ ]:
# Chạy thử với v2: search (an toàn) → chạy luôn, send_email (nguy hiểm) → dừng chờ duyệt

config_v2 = {"configurable": {"thread_id": "hil-demo-v2"}}

print("👉 Test 1: Search database (safe tool) - sẽ chạy tự do\n")
for event in app_v2.stream(
    {"messages": [HumanMessage(content="Search for customer Jane Smith in database")]},
    config=config_v2,
    stream_mode="values",
):
    msg = event["messages"][-1]
    if isinstance(msg, AIMessage) and msg.content:
        print(f"🤖 Agent: {msg.content[:200]}")
    elif isinstance(msg, ToolMessage):
        print(f"🔧 Result: {msg.content[:200]}")
    elif isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🔧 Tool call: {tc['name']}")

state_v2 = app_v2.get_state(config_v2)
print(f"\n📌 State: {'✅ Hoàn thành' if not state_v2.next else '⏸️ Interrupted'}")


In [ ]:
# Test 2: Send email (dangerous tool) → sẽ bị interrupt chờ duyệt

print("👉 Test 2: Send email (dangerous tool) - sẽ bị interrupt\n")
for event in app_v2.stream(
    {"messages": [HumanMessage(content="Send email to john@example.com with subject 'Hello' and body 'Test'")]},
    config=config_v2,
    stream_mode="values",
):
    msg = event["messages"][-1]
    if isinstance(msg, AIMessage) and msg.content:
        print(f"🤖 Agent: {msg.content[:200]}")
    elif isinstance(msg, AIMessage) and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"🔧 Tool call: {tc['name']}({tc['args']})")

state_v2 = app_v2.get_state(config_v2)
print(f"\n📌 State: {'✅ Hoàn thành' if not state_v2.next else '⏸️ Interrupted - chờ bạn duyệt!'}")

if state_v2.next:
    print("\n👉 Chạy cell tiếp theo để duyệt tool call này.")


In [ ]:
# Approve tool call đang chờ (nếu có)
state_v2 = app_v2.get_state(config_v2)

if state_v2.next:
    print("⏸️  Có tool call đang chờ duyệt:")
    last = state_v2.values["messages"][-1]
    for tc in last.tool_calls:
        print(f"  • {tc['name']}({tc['args']})")

    choice = input("\nBạn có muốn approve? (y/n): ").strip().lower()

    if choice == "y":
        print("\n✅ Approving... tiếp tục agent...\n")
        for event in app_v2.stream(None, config=config_v2, stream_mode="values"):
            msg = event["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"🤖 Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"🔧 Tool result: {msg.content[:150]}")
        print("\n✅ Hoàn thành!")
    else:
        print("\n❌ Rejected! Gửi feedback...")
        app_v2.update_state(
            config_v2,
            {"messages": [HumanMessage(content="I reject that. Don't send any emails.")]},
        )
        for event in app_v2.stream(None, config=config_v2, stream_mode="values"):
            msg = event["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"🤖 Agent: {msg.content}")
else:
    print("ℹ️  Không có tool nào chờ duyệt.")


## 8. Log & Audit Trail

Một phần quan trọng của HIL là **audit trail** — ghi lại lịch sử tương tác
giữa human và agent để phục vụ compliance và debugging.

In [ ]:
import pandas as pd
from pathlib import Path

LOG_FILE = Path("hil_audit_log.jsonl")


def log_interaction(thread_id: str, decision: str, tool_name: str, tool_args: dict, reason: str = ""):
    """Ghi log audit trail."""
    entry = {
        "timestamp": datetime.now().isoformat(),
        "thread_id": thread_id,
        "decision": decision,       # approved / rejected / auto
        "tool_name": tool_name,
        "tool_args": tool_args,
        "reason": reason,
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")
    return entry


def view_audit_log():
    """Xem audit log dưới dạng DataFrame."""
    if not LOG_FILE.exists():
        print("📭 Chưa có log nào.")
        return

    records = []
    with open(LOG_FILE) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    df = pd.DataFrame(records)
    if not df.empty:
        print(f"📊 **Audit Log** — {len(df)} entries\n")
        display(df)
    return df


# Ghi thử một vài entries
log_interaction("hil-demo-1", "auto_run", "search_database",
                {"query": "John Doe"}, "Safe tool, auto-approved")
log_interaction("hil-demo-1", "approved", "send_email",
                {"to": "john@example.com", "subject": "Welcome"},
                "User confirmed via HIL panel")

print("✅ Đã ghi 2 entries mẫu vào audit log.")
print(f"📁 File: {LOG_FILE.resolve()}")


In [ ]:
# Xem audit log
view_audit_log()


## 9. Tổng Kết & Best Practices

### 🎯 Những gì bạn đã học

| Concept | Mô Tả |
|---------|-------|
| **Interrupt Points** | `interrupt_before` / `interrupt_after` để dừng graph |
| **Checkpointer** | `MemorySaver` lưu state để resume |
| **Resume** | `app.stream(None, config)` để tiếp tục sau interrupt |
| **Update State** | `app.update_state()` để inject human feedback |
| **Audit Trail** | Ghi log mọi decision để compliance |

### ⚡ Best Practices

1. **Interrupt ít thôi** — Đừng dừng ở mọi tool. Chỉ dừng tool **nguy hiểm** (gửi email, xoá data, thanh toán).
2. **Provide context rõ ràng** — Cho người duyệt thấy đủ thông tin để quyết định.
3. **Timeout** — Set timeout cho approval để tránh treo vô hạn.
4. **Audit trail** — Luôn ghi log ai approved cái gì, lúc nào.
5. **Fallback** — Nếu hết timeout → auto reject hoặc escalate.

### 📚 Liên Kết

- [LangGraph HIL Docs](../concepts/human-in-the-loop.md)
- [Managed Deep Agents](../concepts/managed-deep-agents.md)
- [Checkpointing](../concepts/checkpointing.md)